# testing on tmQMg structures

In [ ]:
import os

import rxembed as rx
import xyzrender as xr

# a tmQMg clone with tmQMg_xyz.zip unzipped
# tmQMg_path = "/path/to/tmQMg/data/xyz/" 
tmQMg_path = "../../../tmQMg/data/xyz/" 


In [ ]:
import random
import os
from rdkit import Chem
import numpy as np
from rdkit.Numerics import rdAlignment

# xyz = 'MACDUQ.xyz'

xyz = random.choice([f for f in os.listdir(tmQMg_path) if f.endswith('.xyz')])

print(f'Using {xyz}')
# display(xr.render(f'{tmQMg_path}{xyz}', bo=False))

struct = rx.read_xyz(f'{tmQMg_path}{xyz}', 0, bond_orders='xyz2mol')
display(struct)

def get_ML_rmsd(structure, reference, iso):
      site = rx.identity(Chem.Mol(iso, True)).centres[0]
      groups = [[site.atom]] + [list(v.atoms) for v in site.vertices if v is not None]
      xyz = [m.GetConformer().GetPositions() for m in (structure, reference)]
      p, q = [np.array([x[list(g)].mean(axis=0) for g in groups]) for x in xyz]
      ssd, _ = rdAlignment.GetAlignmentTransform(p, q, reflect=False)
      return np.sqrt(ssd / len(groups))

try:
    cxsmi = rx.cxsmiles(struct)
    print(f'CX SMI: {cxsmi}')

    # smi = rx.smiles(struct)
    # print(f'SMI: {smi}')

    isos = rx.isomers(struct)
    print(isos.summary())

    for iso in isos:
        cx_iso = rx.cxsmiles(iso)
        ens = rx.embed(iso, n=1, seed=31)
        print(f"M–L RMSD = {get_ML_rmsd(ens.mol, struct, iso):.3f} Å")
        cx = rx.cxsmiles(ens.mol)
        if not cx == cx_iso:
            print(f'Warning, CXSMILES changed in embedding for  {cx_iso}')
        if cx == cxsmi:
            print(f'CXSMILES match for  {cx_iso}')
            display(xr.render(ens.mol, overlay=struct, align_atoms='M,L', bo=False))
            # display(xr.render_gif(xr.load(ens.mol), gif_rot='y'))
        else:
            print(f'CXSMILES mismatch for  {cx_iso}')
            # display(xr.render(ens.mol, overlay=struct, align_atoms='M,L', bo=False))

except Exception as e:
    print(f'Error: {e}')
    display(xr.render(struct, bo=False))
